In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.signal import butter, sosfilt
import pyarrow.parquet as pq
import timm
from tqdm.notebook import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
if os.path.isdir('/kaggle/input/hms-harmful-brain-activity-classification'):
    COMP_DIR = '/kaggle/input/hms-harmful-brain-activity-classification'
else:
    COMP_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

device = (
    torch.device('cuda') if torch.cuda.is_available() else
    torch.device('mps') if torch.backends.mps.is_available() else
    torch.device('cpu')
)
print(f'device: {device}')
print(f'COMP_DIR: {COMP_DIR}')

In [ ]:
# ── Find checkpoints ──────────────────────────────────────────────────────────
import pathlib

CKPT_SEARCH_ROOTS = [
    '/kaggle/input',
    str(pathlib.Path(os.getcwd()).parent / 'checkpoints'),
]

def find_coolz_ckpts():
    found = []
    for root in CKPT_SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for r, _, files in os.walk(root):
            for f in files:
                if f.endswith('.ckpt') and 'coolz' in (r + f).lower():
                    found.append(os.path.join(r, f))
    return sorted(found)

CKPTS = find_coolz_ckpts()
print(f'coolz checkpoints: {len(CKPTS)}')
for c in CKPTS:
    print(' ', c)
if not CKPTS:
    print('  (не знайдено — додай датасет з вагами або перевір шлях)')

In [ ]:
# ── Preprocessing (coolz/dataset.py) ─────────────────────────────────────────

FS = 200
WIN_SAMPLES = 10_000
MICRO = 10
N_CH = 16
N_MACRO = WIN_SAMPLES // MICRO   # 1000
CROP_LENGTHS = [2000, 5000, 10_000]   # 10 s / 25 s / 50 s
VOTE_COLS = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']

DOUBLE_BANANA = [
    ('Fp1', 'F7'), ('F7', 'T3'), ('T3', 'T5'), ('T5', 'O1'),
    ('Fp2', 'F8'), ('F8', 'T4'), ('T4', 'T6'), ('T6', 'O2'),
    ('Fp1', 'F3'), ('F3', 'C3'), ('C3', 'P3'), ('P3', 'O1'),
    ('Fp2', 'F4'), ('F4', 'C4'), ('C4', 'P4'), ('P4', 'O2'),
]
EEG_COLS = list(dict.fromkeys(c for pair in DOUBLE_BANANA for c in pair))
LR_FLIP = [4, 5, 6, 7, 0, 1, 2, 3, 12, 13, 14, 15, 8, 9, 10, 11]

_SOS = butter(5, [0.5, 20.0], btype='bandpass', fs=FS, output='sos')


def load_signal(eeg_id: int, offset_sec: float, eeg_dir: str) -> np.ndarray:
    """Returns (16, 10000) float32 bandpass-filtered signal."""
    raw = pq.read_table(f'{eeg_dir}/{eeg_id}.parquet', columns=EEG_COLS).to_pandas()
    start = int(offset_sec * FS)
    chunk = raw.iloc[start: start + WIN_SAMPLES]
    sig = np.stack(
        [chunk[a].values - chunk[b].values for a, b in DOUBLE_BANANA], axis=0
    ).astype(np.float32)
    if sig.shape[1] < WIN_SAMPLES:
        sig = np.pad(sig, ((0, 0), (0, WIN_SAMPLES - sig.shape[1])))
    sig = np.nan_to_num(sig, nan=0.0, posinf=1024.0, neginf=-1024.0)
    sig = np.clip(sig, -1024.0, 1024.0) / 32.0
    return sosfilt(_SOS, sig, axis=-1).astype(np.float32)


def signals_to_image(sig: np.ndarray, center: int | None = None) -> np.ndarray:
    """(16, 10000) → (3, 160, 1000)  —  3 temporal crops as RGB channels."""
    IMG_H = N_CH * MICRO  # 160
    T = sig.shape[1]
    if center is None:
        center = T // 2
    channels = []
    for crop_len in CROP_LENGTHS:
        if crop_len >= T:
            crop = sig
        else:
            s = int(np.clip(center - crop_len // 2, 0, T - crop_len))
            crop = sig[:, s: s + crop_len]
        n = (crop.shape[1] // MICRO) * MICRO
        frame = (crop[:, :n]
                 .reshape(N_CH, n // MICRO, MICRO)
                 .transpose(0, 2, 1)
                 .reshape(IMG_H, n // MICRO)
                 .astype(np.float32))
        if frame.shape[1] != N_MACRO:
            frame = cv2.resize(frame, (N_MACRO, IMG_H), interpolation=cv2.INTER_LINEAR)
        channels.append(frame)
    img = np.stack(channels)  # (3, 160, 1000)
    return (img - img.mean()) / (img.std() + 1e-6)

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────

class CoolzTestDataset(Dataset):
    """
    Returns (img_orig, img_flip) for TTA.
    Both are (3, 160, 1000); img_flip uses LR-flipped signal.
    """
    def __init__(self, df, eeg_dir):
        self.df = df.reset_index(drop=True)
        self.eeg_dir = eeg_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sig = load_signal(
            int(row['eeg_id']),
            float(row.get('eeg_label_offset_seconds', 0)),
            self.eeg_dir,
        )
        img_orig = signals_to_image(sig)
        img_flip = signals_to_image(sig[LR_FLIP])
        return torch.from_numpy(img_orig), torch.from_numpy(img_flip)

In [ ]:
# ── Model (coolz/model.py) ────────────────────────────────────────────────────

class EEGNet(nn.Module):
    def __init__(self, backbone='efficientnet_b5', dropout=0.5):
        super().__init__()
        self.net = timm.create_model(
            backbone, pretrained=False, in_chans=3,
            num_classes=0, global_pool='',
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(self.net.num_features, 6)

    def forward(self, x):
        x = self.net(x)
        x = self.pool(x).view(x.size(0), -1)
        x = self.drop(x)
        return F.log_softmax(self.fc(x), dim=1)


FALLBACK_BACKBONE = 'efficientnet_b5'  # для старих чекпоінтів без метадати


def load_model(ckpt_path):
    raw = torch.load(ckpt_path, map_location=device, weights_only=True)
    if isinstance(raw, dict) and 'state_dict' in raw:
        backbone = raw.get('backbone', FALLBACK_BACKBONE)
        dropout = raw.get('dropout', 0.5)
        state = raw['state_dict']
    else:
        backbone, dropout, state = FALLBACK_BACKBONE, 0.5, raw
    model = EEGNet(backbone, dropout).to(device)
    model.load_state_dict(state)
    model.eval()
    print(f'  loaded [{backbone}]: {os.path.basename(ckpt_path)}')
    return model

In [ ]:
# ── Config + data ─────────────────────────────────────────────────────────────
BATCH_SIZE = 8

test_df = pd.read_csv(os.path.join(COMP_DIR, 'test.csv'))
sub = pd.read_csv(os.path.join(COMP_DIR, 'sample_submission.csv'))

if 'eeg_label_offset_seconds' not in test_df.columns:
    test_df['eeg_label_offset_seconds'] = 0.0

eeg_dir = os.path.join(COMP_DIR, 'test_eegs')
ds = CoolzTestDataset(test_df, eeg_dir)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'test samples : {len(test_df)}')
print(f'checkpoints  : {len(CKPTS)}')
if not CKPTS:
    raise RuntimeError('Не знайдено coolz чекпоінтів.')

In [ ]:
# ── Inference: ensemble + TTA (LR flip) ──────────────────────────────────────

all_fold_probs = []

for ckpt_path in CKPTS:
    print(f'\n→ {os.path.basename(ckpt_path)}')
    model = load_model(ckpt_path)

    probs_orig, probs_flip = [], []
    with torch.no_grad():
        for img, img_flip in tqdm(loader, desc='inference', leave=False):
            probs_orig.append(model(img.to(device)).exp().cpu().numpy())
            probs_flip.append(model(img_flip.to(device)).exp().cpu().numpy())

    p = (np.concatenate(probs_orig) + np.concatenate(probs_flip)) / 2
    all_fold_probs.append(p)
    print(f'  mean pred: {p.mean(axis=0).round(3)}')

final_probs = np.mean(all_fold_probs, axis=0)
print(f'\nEnsemble: {len(all_fold_probs)} checkpoint(s)')
print(f'final mean: {final_probs.mean(axis=0).round(3)}')

In [ ]:
# ── Submission ────────────────────────────────────────────────────────────────
sub[VOTE_COLS] = final_probs
sub.to_csv('submission.csv', index=False)
print('Saved: submission.csv')
print(sub.head())